# Extract Graphormer Embeddings for DA Datasets

Extract 768-d Graphormer embeddings for every molecule in the CV datasets

- collect SMILES from the source/target of every fold
- extract embeddings with Graphormer (768-d dense vectors)
- save in npz format {smiles -> vector}

In [14]:
import argparse
import os
import sys

import numpy as np
import pandas as pd
import torch

# ============================================================
# Project paths
# ============================================================
ROOT = r".."

# Add src to the Python import path
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

CV_DATASETS = os.path.join(ROOT, "data", "cv_datasets")
FEAT = os.path.join(ROOT, "features")

os.makedirs(FEAT, exist_ok=True)

# ============================================================
# Settings
# ============================================================
PROPERTIES = ["clearance", "half_life", "fu"]
N_FOLDS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_NODES = 128

print(f"Device: {DEVICE}")
print(f"Root: {ROOT}")
print(f"CV Datasets: {CV_DATASETS}")
print(f"Features: {FEAT}")
print(f"Source: {SRC}")

# ============================================================
# Check OGB
# ============================================================
try:
    import ogb
    print(f"OGB: OK ({ogb.__version__})")
except ModuleNotFoundError:
    print("ERROR: OGB is not installed in the current Python environment.")
    print(f"Python: {sys.executable}")
    print("Install with:")
    print(f'"{sys.executable}" -m pip install ogb')
    raise

Device: cpu
Root: ..
CV Datasets: ..\data\cv_datasets
Features: ..\features
Source: ..\src
OGB: OK (1.3.6)


## Step 1: Load the Graphormer encoder

In [15]:
# TODO: import the project's graphormer_encoder
# from src.graphormer_encoder import embed, load_encoder
# from src.graphormer_preprocess import collate, preprocess_smiles

# Placeholder for now - the import above must be fixed in practice
try:
    sys.path.insert(0, ROOT)
    from src.graphormer_encoder import embed, load_encoder
    from src.graphormer_preprocess import collate, preprocess_smiles
    print("✓ Graphormer encoder loaded")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nNote: check the path of the project's src.graphormer_encoder/preprocess")

✓ Graphormer encoder loaded


# TODO: import the project's graphormer_encoder
sys.path.insert(0, ROOT)

try:
    from src.graphormer_encoder import embed, load_encoder
    from src.graphormer_preprocess import collate, preprocess_smiles
    print("✓ Graphormer encoder loaded successfully")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nTroubleshooting:")
    print("  1. Check src/ directory exists")
    print("  2. Verify graphormer_encoder.py and graphormer_preprocess.py exist")
    print("  3. Check function names: load_encoder, embed, collate, preprocess_smiles")

In [16]:
def collect_all_smiles():
    """Collect SMILES from the source/target of every fold"""
    smiles_set = set()
    
    for fold_idx in range(N_FOLDS):
        fold_dir = os.path.join(CV_DATASETS, f'fold_{fold_idx}')
        
        # source/target_train/target_test files for each property
        files = []
        for prop in PROPERTIES:
            files.append(os.path.join(fold_dir, f'{prop}_source.csv'))
            files.append(os.path.join(fold_dir, f'{prop}_target_train.csv'))
            files.append(os.path.join(fold_dir, f'{prop}_target_test.csv'))
        
        for file_path in files:
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                # Handle the SMILES column name flexibly
                if 'smiles' in df.columns:
                    smiles_list = df['smiles'].astype(str).tolist()
                elif 'Drug' in df.columns:
                    smiles_list = df['Drug'].astype(str).tolist()
                else:
                    print(f"  ⚠️ No SMILES column in {file_path}, columns: {list(df.columns)}\"")
                    continue
                smiles_set.update(smiles_list)
    
    return sorted(smiles_set)

print("Collecting SMILES from all CV folds...")
all_smiles = collect_all_smiles()
print(f"Total unique molecules: {len(all_smiles)}")

~\AppData\Local\Temp\ipykernel_15452\1353432180.py:17: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
~\AppData\Local\Temp\ipykernel_15452\1353432180.py:17: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
~\AppData\Local\Temp\ipykernel_15452\1353432180.py:17: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
~\AppData\Local\Temp\ipykernel_15452\1353432180.py:17: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Total unique molecules: 51338


~\AppData\Local\Temp\ipykernel_15452\1353432180.py:17: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


## Step 3: Extract Graphormer embeddings

In [17]:
def extract_embeddings(smiles_list, batch_size=16):
    """Extract Graphormer embeddings for every molecule"""
    
    # Load the encoder
    try:
        enc, cfg = load_encoder(DEVICE)
    except Exception as e:
        print(f"Error loading encoder: {e}")
        return None, None
    
    keys = []  # SMILES list
    vecs = []  # embedding list
    buf_items = []  # current batch
    buf_keys = []
    n_fail = 0
    
    print(f"Extracting embeddings for {len(smiles_list)} molecules...\n")
    
    def flush():
        nonlocal buf_items, buf_keys
        if not buf_items:
            return
        try:
            batch = collate(buf_items, spatial_pos_max=cfg.spatial_pos_max)
            emb = embed(enc, batch, DEVICE).numpy().astype('float32')
            keys.extend(buf_keys)
            vecs.append(emb)
        except Exception as e:
            print(f"  Batch processing error: {e}")
        buf_items, buf_keys = [], []
    
    for i, smi in enumerate(smiles_list):
        try:
            # Preprocess SMILES
            item = preprocess_smiles(smi)
            if item is None or item['num_nodes'] > MAX_NODES:
                n_fail += 1
                continue
            
            buf_items.append(item)
            buf_keys.append(smi)
            
        except Exception:
            n_fail += 1
            continue
        
        # Flush buffer when full
        if len(buf_items) >= batch_size:
            flush()
        
        # Progress
        if (i + 1) % 2000 == 0:
            print(f"  {i + 1}/{len(smiles_list)}  (embedded {len(keys)}, failed {n_fail})")
    
    # Final flush
    flush()
    
    # Concatenate all embeddings
    X = np.concatenate(vecs, axis=0) if vecs else np.zeros((0, 768), dtype='float32')
    
    print(f"\n✓ Extracted {len(keys)} embeddings (failed/skipped {n_fail})")
    return keys, X

# Extract embeddings (note: needs GPU memory)
# keys, embeddings = extract_embeddings(all_smiles, batch_size=16)

## Step 4: Save

In [21]:
def save_embeddings(keys, embeddings, output_path):
    """Save embeddings in npz format"""
    if keys is None or embeddings is None:
        print("No embeddings to save")
        return
    
    np.savez_compressed(
        output_path,
        keys=np.array(keys),
        X=embeddings
    )
    print(f"✓ Saved {len(keys)} embeddings -> {output_path}")

# Save
output_path = os.path.join(FEAT, 'graphormer_cv_embeddings.npz')
# save_embeddings(keys, embeddings, output_path)

## Step 5: Validation

In [20]:
def load_and_verify(npz_path):
    """Load and validate the saved embeddings"""
    if not os.path.exists(npz_path):
        print(f"File not found: {npz_path}")
        return
    
    data = np.load(npz_path, allow_pickle=True)
    keys = data['keys']
    X = data['X']
    
    print(f"\nEmbedding Bank Info:")
    print(f"  Total molecules: {len(keys)}")
    print(f"  Embedding shape: {X.shape}")
    print(f"  Embedding dim: {X.shape[1]}")
    print(f"  Data type: {X.dtype}")
    print(f"\n  Example SMILES -> 768d vector:")
    for i in range(min(3, len(keys))):
        print(f"    {keys[i][:50]}... : {X[i][:5]}...")

# Verify
# load_and_verify(output_path)

## Execution flow

```python
# 1. Collect all SMILES
all_smiles = collect_all_smiles()  # a few thousand molecules

# 2. Extract Graphormer embeddings (takes a while)
keys, embeddings = extract_embeddings(all_smiles, batch_size=16)

# 3. Save
output_path = os.path.join(FEAT, 'graphormer_cv_embeddings.npz')
save_embeddings(keys, embeddings, output_path)

# 4. Validate
load_and_verify(output_path)
```

## Notes

- needs GPU memory (batch_size is adjustable)
- the model is downloaded automatically on first run
- embedding extraction takes time (minutes to tens of minutes)
- the npz file is used for CV training